# 🔬 NIDS - Cahier de Recherche & Exploration R&D (CIC-IDS-2017)

> **Note Importante pour la Production :**  
> Ce notebook (`NIDS.ipynb`) documente les étapes historiques d'exploration des captures PCAP brutes, du reverse-engineering des protocoles avec NFStream et de la découverte des biais de classification.  
>  
> Pour le **pipeline d'entraînement officiel de production (v5)**, validé sans duplication de features (58 strictes), avec **calibration de température (T=1.4982)**, benchmark **LightGBM**, **Auto-Encodeur v2** et **suites de tests automatisées (32 tests)**, utilisez directement :  
> 👉 **`retrain_dnn_v5_colab.ipynb`** et consultez le registre centralisé **`models/manifest.json`**.

---

## 📑 Sommaire des 4 Blocs R&D

1. **Bloc 1 : Ingestion & Extraction NFStream (CIC-IDS-2017)** — Montage Drive, configuration NFStreamer, extraction des flux bidirectionnels (Cellules 1 à 21).
2. **Bloc 2 : Étiquetage Temporel & Feature Engineering (58 Dimensions)** — Règles de corrélation temporelle, sélection des variables statistiques L4/L7, normalisation robuste (Cellules 22 à 40).
3. **Bloc 3 : Modélisation Initiale & Diagnostic des Faux Positifs** — Entraînement baseline Random Forest vs DNN PyTorch, analyse d'erreurs sur les flux limites (Cellules 41 à 60).
4. **Bloc 4 : Analyse Exploratoire Multi-Jours (Lundi au Vendredi)** — Traitement chronologique complet, détection DoS/DDoS/PortScan, consolidation de `dataset_5jours.parquet` (Cellules 61 à 85).

---




# 1.   Montage de Drive



In [ ]:
# Monter Google Drive (une fenêtre te demandera d'autoriser l'accès)
from google.colab import drive
drive.mount('/content/drive')

# Créer un dossier de travail pour le projet
import os
os.makedirs('/content/drive/MyDrive/nids_project/pcap', exist_ok=True)
os.makedirs('/content/drive/MyDrive/nids_project/features', exist_ok=True)
os.makedirs('/content/drive/MyDrive/nids_project/models', exist_ok=True)



# 2.   Installation de NFStream



In [ ]:
!pip install nfstream

#3.   Test d'extraction



In [ ]:
from nfstream import NFStreamer

streamer = NFStreamer(source="/content/drive/MyDrive/nids_project/pcap/dos.pcap",
                      statistical_analysis=True, n_dissections=0)
df = streamer.to_pandas()

print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
print(len(num_cols), "colonnes numériques")
print(num_cols)



# 4.   Netoyage du features



In [ ]:
cols_a_retirer = [
    'id', 'expiration_id', 'src_port', 'dst_port',
    'vlan_id', 'tunnel_id', 'ip_version',
    'bidirectional_first_seen_ms', 'bidirectional_last_seen_ms',
    'src2dst_first_seen_ms', 'src2dst_last_seen_ms',
    'dst2src_first_seen_ms', 'dst2src_last_seen_ms',
    'application_is_guessed', 'application_confidence',
    'requested_server_name', 'client_fingerprint',
    'server_fingerprint', 'user_agent', 'content_type'
]

feature_cols = [c for c in num_cols if c not in cols_a_retirer]
print(len(feature_cols), "features retenues")
print(feature_cols)

# Sauvegarder cette liste : elle devra être IDENTIQUE à l'entraînement et en production
import json
with open('/content/drive/MyDrive/nids_project/models/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)



# 5.   Telechargement du datasets



In [ ]:
from huggingface_hub import list_repo_files

for f in list_repo_files("bencorn/CICIDS2017", repo_type="dataset"):
    print(f)

In [ ]:
!ls -lh /content/pcap

In [ ]:
from huggingface_hub import hf_hub_download

local_path = hf_hub_download(
    repo_id="bencorn/CICIDS2017",
    repo_type="dataset",
    filename="pcaps/Tuesday-WorkingHours.pcap",
    local_dir="/content/pcap"
)
print(local_path)

In [ ]:
!ls -lh /content/pcap/pcaps
!df -h /content



# 6.   Extraction du features




In [ ]:
from nfstream import NFStreamer

pcap = "/content/pcap/pcaps/Tuesday-WorkingHours.pcap"
out_csv = "/content/tuesday_flows.csv"

streamer = NFStreamer(
    source=pcap,
    statistical_analysis=True,   # tailles, délais, etc.
    n_dissections=0,             # désactive nDPI (inutile et lent)
    idle_timeout=120,            # un flux inactif 120 s est terminé
    active_timeout=1800,         # un flux de plus de 30 min est coupé
    n_meters=0                   # 0 = utilise tous les cœurs disponibles
)

n = streamer.to_csv(path=out_csv)
print(n, "flux extraits")

In [ ]:
!cp /content/tuesday_flows.csv /content/drive/MyDrive/nids_project/features/tuesday_flows.csv

In [ ]:
import pandas as pd

df = pd.read_csv(out_csv)
print(df.shape)
print(df["src_ip"].value_counts().head(10))
print(df["dst_ip"].value_counts().head(10))

In [ ]:
import pandas as pd

df = pd.read_csv("/content/tuesday_flows.csv")
df["start"] = pd.to_datetime(df["bidirectional_first_seen_ms"], unit="ms")

cand = df[(df.src_ip == "172.16.0.1") & (df.dst_ip == "192.168.10.50")]

for port in [21, 22]:
    sub = cand[cand.dst_port == port]
    print(f"\n=== Port {port} : {len(sub)} flux ===")
    print("premier :", sub["start"].min())
    print("dernier :", sub["start"].max())
    print(sub["start"].dt.floor("10min").value_counts().sort_index())

# **Etuquetage**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/tuesday_flows.csv")
df["start"] = pd.to_datetime(df["bidirectional_first_seen_ms"], unit="ms")

# Fenêtres en UTC, avec ~5 min de marge de chaque côté des pics observés
ftp_start, ftp_end = pd.Timestamp("2017-07-04 12:15"), pd.Timestamp("2017-07-04 13:25")
ssh_start, ssh_end = pd.Timestamp("2017-07-04 17:05"), pd.Timestamp("2017-07-04 18:20")

is_att = (df.src_ip == "172.16.0.1") & (df.dst_ip == "192.168.10.50")

df["label"] = "BENIGN"
df.loc[is_att & (df.dst_port == 21) & df.start.between(ftp_start, ftp_end), "label"] = "FTP-Patator"
df.loc[is_att & (df.dst_port == 22) & df.start.between(ssh_start, ssh_end), "label"] = "SSH-Patator"

print(df["label"].value_counts())

Verification

In [ ]:
# Flux de l'attaquant vers la victime que l'on a laissés en BENIGN : y a-t-il un oubli ?
reste = df[is_att & (df.label == "BENIGN")]
print(len(reste), "flux 172.16.0.1 -> 192.168.10.50 restés BENIGN")
print(reste["dst_port"].value_counts().head(10))

Sauvegarde

In [ ]:
df.to_csv("/content/drive/MyDrive/nids_project/features/tuesday_labeled.csv", index=False)

charger, sélectionner les features, nettoyer

In [ ]:
import json, numpy as np, pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids_project/features/tuesday_labeled.csv")
df["start"] = pd.to_datetime(df["start"])

with open("/content/drive/MyDrive/nids_project/models/feature_cols.json") as f:
    feature_cols = json.load(f)

# Vérifier que toutes les colonnes existent bien dans ce CSV
manquantes = [c for c in feature_cols if c not in df.columns]
print("Colonnes manquantes :", manquantes)
print(len(feature_cols), "features")

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["label"]

split par blocs de temps

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["start"].dt.floor("5min").astype(str)

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("TRAIN\n", y_train.value_counts())
print("TEST\n", y_test.value_counts())

baseline Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                            n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print(classification_report(y_test, pred, digits=4))
print(confusion_matrix(y_test, pred))

Quelles features comptent ?

In [ ]:
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(imp.head(15))

Pourquoi ces 20 flux sont-ils ratés ?

In [ ]:
test = df.iloc[test_idx].copy()
test["pred"] = pred
rates = test[(test.label != "BENIGN") & (test.pred == "BENIGN")]

print(rates[["label", "protocol", "bidirectional_packets", "bidirectional_bytes",
             "bidirectional_duration_ms", "bidirectional_syn_packets",
             "bidirectional_rst_packets", "bidirectional_fin_packets"]].describe())

comparer avec les attaques bien détectées

In [ ]:
bien = test[(test.label != "BENIGN") & (test.pred == test.label)]
cols = ["bidirectional_packets", "bidirectional_duration_ms",
        "bidirectional_rst_packets", "bidirectional_fin_packets"]
print(bien[cols].describe())

Bloc 1 : préparation

In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

classes = ["BENIGN", "FTP-Patator", "SSH-Patator"]
label2id = {c: i for i, c in enumerate(classes)}

def prep(X):
    return np.log1p(X.clip(lower=0))   # à réutiliser à l'identique en production

Xtr_full, Xte = prep(X_train), prep(X_test)
ytr_full = y_train.map(label2id).values
yte = y_test.map(label2id).values

# Validation : 20 % du train, par blocs de 5 min
g_tr = groups.iloc[train_idx].values
tr_i, va_i = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                  .split(Xtr_full, ytr_full, g_tr))
print("Classes dans la validation :", np.bincount(ytr_full[va_i], minlength=3))

scaler = StandardScaler().fit(Xtr_full.iloc[tr_i])   # fit sur le train seulement
Xtr = scaler.transform(Xtr_full.iloc[tr_i]); ytr = ytr_full[tr_i]
Xva = scaler.transform(Xtr_full.iloc[va_i]); yva = ytr_full[va_i]
Xte_s = scaler.transform(Xte)

joblib.dump(scaler, "/content/drive/MyDrive/nids_project/models/scaler.joblib")

Bloc 2 : le DNN et son entraînement (PyTorch)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"
T = lambda a, dt: torch.tensor(np.asarray(a), dtype=dt)

loader = DataLoader(TensorDataset(T(Xtr, torch.float32), T(ytr, torch.long)),
                    batch_size=1024, shuffle=True)

model = nn.Sequential(
    nn.Linear(Xtr.shape[1], 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 64),           nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 32),            nn.ReLU(),
    nn.Linear(32, 3)
).to(device)

w = compute_class_weight("balanced", classes=np.arange(3), y=ytr)
loss_fn = nn.CrossEntropyLoss(weight=T(w, torch.float32).to(device))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def predict(X):
    model.eval()
    with torch.no_grad():
        return model(T(X, torch.float32).to(device)).argmax(1).cpu().numpy()

best, patience = 0, 0
for epoch in range(40):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
    f1 = f1_score(yva, predict(Xva), average="macro")
    print(f"epoch {epoch+1:2d}  loss {loss.item():.4f}  val F1 macro {f1:.4f}")
    if f1 > best:
        best, patience = f1, 0
        torch.save(model.state_dict(), "/content/drive/MyDrive/nids_project/models/dnn.pt")
    else:
        patience += 1
        if patience >= 5: break    # arrêt anticipé

model.load_state_dict(torch.load("/content/drive/MyDrive/nids_project/models/dnn.pt"))
pred = predict(Xte_s)
print(classification_report(yte, pred, target_names=classes, digits=4))
print(confusion_matrix(yte, pred))

comprendre les 58 faux positifs

In [ ]:
test["pred_dnn"] = [classes[i] for i in pred]
fp = test[(test.label == "BENIGN") & (test.pred_dnn == "SSH-Patator")]

print(len(fp), "faux positifs")
print(fp["dst_port"].value_counts().head())
print(fp["src_ip"].value_counts().head())
print(fp["dst_ip"].value_counts().head())
print(fp["start"].dt.floor("10min").value_counts().sort_index())

Une vérification pour savoir si le problème est grave

In [ ]:
for port in [22, 443]:
    b = test[(test.label == "BENIGN") & (test.dst_port == port)]
    fpn = (b.pred_dnn == "SSH-Patator").sum()
    print(f"Port {port} : {len(b)} flux bénins dans le test, dont {fpn} classés SSH-Patator")

cols = ["bidirectional_packets", "bidirectional_bytes", "bidirectional_duration_ms",
        "bidirectional_psh_packets", "bidirectional_rst_packets"]
vrai_ssh = test[test.label == "SSH-Patator"]
print("\nFaux positifs :\n", fp[cols].median())
print("\nVrais SSH-Patator :\n", vrai_ssh[cols].median())

Vérification et expérience en une seule cellule

In [ ]:
tr = df.iloc[train_idx]
print("SSH bénins dans le train :", ((tr.label == "BENIGN") & (tr.dst_port == 22)).sum())

def train_dnn(weights, seed=0, max_epochs=40):
    torch.manual_seed(seed); np.random.seed(seed)
    m = nn.Sequential(
        nn.Linear(Xtr.shape[1], 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 3)).to(device)
    lf = nn.CrossEntropyLoss(weight=None if weights is None
                             else T(weights, torch.float32).to(device))
    o = torch.optim.Adam(m.parameters(), lr=1e-3)
    best, pat, state = 0, 0, None
    for ep in range(max_epochs):
        m.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            o.zero_grad(); lf(m(xb), yb).backward(); o.step()
        m.eval()
        with torch.no_grad():
            pv = m(T(Xva, torch.float32).to(device)).argmax(1).cpu().numpy()
        f1 = f1_score(yva, pv, average="macro")
        if f1 > best:
            best, pat = f1, 0
            state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            pat += 1
            if pat >= 5: break
    m.load_state_dict(state); m.eval()
    with torch.no_grad():
        p = m(T(Xte_s, torch.float32).to(device)).argmax(1).cpu().numpy()
    return p

w_bal = compute_class_weight("balanced", classes=np.arange(3), y=ytr)
for nom, w in [("balanced", w_bal), ("racine carrée", np.sqrt(w_bal)), ("aucun", None)]:
    p = train_dnn(w)
    cm = confusion_matrix(yte, p)
    fn = cm[1, 0] + cm[2, 0]          # attaques classées BENIGN
    fp = cm[0, 1] + cm[0, 2]          # normal classé attaque
    print(f"{nom:14s} F1 macro {f1_score(yte, p, average='macro'):.4f} | faux négatifs {fn} | faux positifs {fp}")

Script de téléchargement et d’extraction du lundi

In [ ]:
# 0. (Si la session a été réinitialisée) réinstaller et remonter Drive
!pip install -q nfstream huggingface_hub
from google.colab import drive
drive.mount('/content/drive')

# 1. Télécharger le lundi (11 Go)
from huggingface_hub import hf_hub_download
local_path = hf_hub_download(
    repo_id="bencorn/CICIDS2017",
    repo_type="dataset",
    filename="pcaps/Monday-WorkingHours.pcap",
    local_dir="/content/pcap"
)
print(local_path)

In [ ]:
# 2. Extraire les flux avec les MÊMES paramètres que le mardi
from nfstream import NFStreamer

pcap = "/content/pcap/pcaps/Monday-WorkingHours.pcap"
out_csv = "/content/monday_flows.csv"

streamer = NFStreamer(
    source=pcap,
    statistical_analysis=True,
    n_dissections=0,
    idle_timeout=120,
    active_timeout=1800,
    n_meters=0
)
n = streamer.to_csv(path=out_csv)
print(n, "flux extraits")

In [ ]:
# 3. Étiqueter (tout est BENIGN), sauvegarder sur Drive, libérer le disque
import pandas as pd

df_m = pd.read_csv(out_csv)
df_m["start"] = pd.to_datetime(df_m["bidirectional_first_seen_ms"], unit="ms")
df_m["label"] = "BENIGN"

print(df_m.shape)
print(df_m["start"].min(), "->", df_m["start"].max())

df_m.to_csv("/content/drive/MyDrive/nids_project/features/monday_labeled.csv", index=False)

!rm /content/pcap/pcaps/Monday-WorkingHours.pcap

Bloc 1 : fusion, split, préparation

In [ ]:
import json, joblib, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

base = "/content/drive/MyDrive/nids_project"
df = pd.concat([pd.read_csv(f"{base}/features/monday_labeled.csv"),
                pd.read_csv(f"{base}/features/tuesday_labeled.csv")], ignore_index=True)
df["start"] = pd.to_datetime(df["start"])

with open(f"{base}/models/feature_cols.json") as f:
    feature_cols = json.load(f)

classes = ["BENIGN", "FTP-Patator", "SSH-Patator"]
label2id = {c: i for i, c in enumerate(classes)}

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
X = np.log1p(X.clip(lower=0))
y = df["label"].map(label2id).values
groups = df["start"].dt.floor("5min").astype(str).values

def split_ok(yy):
    return np.bincount(yy, minlength=3).min() >= 100

# 1) split train / test par blocs de temps ; on change de graine tant qu'il manque une classe
for seed in range(20):
    tr_i, te_i = next(GroupShuffleSplit(1, test_size=0.3, random_state=seed).split(X, y, groups))
    if split_ok(y[te_i]) and split_ok(y[tr_i]): break

# 2) validation = 20 % du train, par blocs de temps
for seed2 in range(20):
    a, b = next(GroupShuffleSplit(1, test_size=0.2, random_state=seed2).split(X.iloc[tr_i], y[tr_i], groups[tr_i]))
    if split_ok(y[tr_i][b]): break
trn_i, val_i = tr_i[a], tr_i[b]

for nom, idx in [("TRAIN", trn_i), ("VAL", val_i), ("TEST", te_i)]:
    print(nom, dict(zip(classes, np.bincount(y[idx], minlength=3))))

scaler = StandardScaler().fit(X.iloc[trn_i])          # ajusté sur le train seulement
Xtr, Xva, Xte = (scaler.transform(X.iloc[i]) for i in (trn_i, val_i, te_i))
ytr, yva, yte = y[trn_i], y[val_i], y[te_i]
joblib.dump(scaler, f"{base}/models/scaler_v2.joblib")   # v2 : on garde l'ancien intact

Bloc 2 : réentraîner et comparer trois réglages, avec trois graines

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

device = "cuda" if torch.cuda.is_available() else "cpu"
T = lambda a, dt: torch.tensor(np.asarray(a), dtype=dt)
loader = DataLoader(TensorDataset(T(Xtr, torch.float32), T(ytr, torch.long)),
                    batch_size=1024, shuffle=True)

def train_dnn(weights, seed, max_epochs=40):
    torch.manual_seed(seed); np.random.seed(seed)
    m = nn.Sequential(
        nn.Linear(Xtr.shape[1], 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 3)).to(device)
    lf = nn.CrossEntropyLoss(weight=None if weights is None
                             else T(weights, torch.float32).to(device))
    o = torch.optim.Adam(m.parameters(), lr=1e-3)
    best, pat, state = -1, 0, None
    for ep in range(max_epochs):
        m.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            o.zero_grad(); lf(m(xb), yb).backward(); o.step()
        m.eval()
        with torch.no_grad():
            pv = m(T(Xva, torch.float32).to(device)).argmax(1).cpu().numpy()
        f1 = f1_score(yva, pv, average="macro")
        if f1 > best:
            best, pat, state = f1, 0, {k: v.clone() for k, v in m.state_dict().items()}
        else:
            pat += 1
            if pat >= 5: break
    m.load_state_dict(state); m.eval()
    with torch.no_grad():
        return m(T(Xte, torch.float32).to(device)).argmax(1).cpu().numpy(), m

w_bal = compute_class_weight("balanced", classes=np.arange(3), y=ytr)
test_df = df.iloc[te_i]

for nom, w in [("balanced", w_bal), ("racine carrée", np.sqrt(w_bal)), ("aucun", None)]:
    for s in range(3):
        p, _ = train_dnn(w, seed=s)
        cm = confusion_matrix(yte, p, labels=[0, 1, 2])
        fn = cm[1, 0] + cm[2, 0]
        fp_mask = (yte == 0) & (p != 0)
        fp_22  = (fp_mask & (test_df.dst_port.values == 22)).sum()
        fp_443 = (fp_mask & (test_df.dst_port.values == 443)).sum()
        print(f"{nom:14s} graine {s} | F1 macro {f1_score(yte, p, average='macro'):.4f} "
              f"| ratées {fn} | fausses alertes {fp_mask.sum()} (port 22 : {fp_22}, port 443 : {fp_443})")

Comparaison honnête avec le Random Forest sur le même split, et sauvegarde du DNN

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                            n_jobs=-1, random_state=42)
rf.fit(Xtr, ytr)
cm = confusion_matrix(yte, rf.predict(Xte), labels=[0, 1, 2])
print("RF : ratées", cm[1, 0] + cm[2, 0], "| fausses alertes", cm[0, 1] + cm[0, 2])

# Sauvegarder le DNN retenu (sans poids de classes)
p, model = train_dnn(None, seed=0)
torch.save(model.state_dict(), f"{base}/models/dnn_v2.pt")

Mercredi


In [ ]:
# Si la session a été réinitialisée
!pip install -q nfstream huggingface_hub
from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import hf_hub_download
local_path = hf_hub_download(
    repo_id="bencorn/CICIDS2017",
    repo_type="dataset",
    filename="pcaps/Wednesday-workingHours.pcap",   # attention : "w" minuscule
    local_dir="/content/pcap"
)
print(local_path)

In [ ]:
!pip install nfstream

In [ ]:
from nfstream import NFStreamer

streamer = NFStreamer(
    source="/content/pcap/pcaps/Wednesday-workingHours.pcap",
    statistical_analysis=True,
    n_dissections=0,
    idle_timeout=120,
    active_timeout=1800,
    n_meters=0
)
n = streamer.to_csv(path="/content/wednesday_flows.csv")
print(n, "flux extraits")

In [ ]:
import pandas as pd

df = pd.read_csv("/content/wednesday_flows.csv")
df["start"] = pd.to_datetime(df["bidirectional_first_seen_ms"], unit="ms")
print(df.shape, df["start"].min(), "->", df["start"].max())

# Sauvegarde brute sur Drive AVANT d'étiqueter (on ne perd rien si la session saute)
df.to_csv("/content/drive/MyDrive/nids_project/features/wednesday_raw.csv", index=False)
!rm /content/pcap/pcaps/Wednesday-workingHours.pcap

In [ ]:
# Qui envoie le plus de flux vers qui ? (hors trafic interne ordinaire)
ext = df[~df.src_ip.str.startswith("192.168.10.")]
print(ext.groupby(["src_ip", "dst_ip", "dst_port"]).size()
         .sort_values(ascending=False).head(15))

# Fenêtres : flux de 172.16.0.1 vers le port 80, par tranche de 10 min
c = df[(df.src_ip == "172.16.0.1") & (df.dst_port == 80)]
print(c["start"].dt.floor("10min").value_counts().sort_index())

# Heartbleed : port 444
h = df[df.dst_port == 444]
print(len(h), "flux vers le port 444")
print(h["start"].dt.floor("10min").value_counts().sort_index())

In [ ]:
a = df[(df.src_ip == "172.16.0.1") & (df.dst_ip == "192.168.10.50") & (df.dst_port == 80)]
pm = a.set_index("start").sort_index().resample("1min").size()
pm = pm[pm > 0]
print(" ".join(f"{t:%H:%M}={n}" for t, n in pm.items()))

In [ ]:
h = df[df.dst_port == 444]
print(h[["start", "src_ip", "dst_ip", "src_port", "dst_port",
         "bidirectional_duration_ms", "bidirectional_packets",
         "bidirectional_bytes"]].to_string())

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids_project/features/wednesday_raw.csv")
df["start"] = pd.to_datetime(df["start"])
d = "2017-07-05 "

fenetres = {                      # [début, fin[  en UTC
    "DoS-slowloris":    ("12:48", "13:11"),
    "DoS-Slowhttptest": ("13:15", "13:36"),
    "DoS-Hulk":         ("13:43", "14:00"),
    "DoS-GoldenEye":    ("14:10", "14:20"),
}

att = df.src_ip == "172.16.0.1"
web = att & (df.dst_ip == "192.168.10.50") & (df.dst_port == 80)

df["label"] = "BENIGN"
for nom, (a, b) in fenetres.items():
    m = web & (df.start >= pd.Timestamp(d + a)) & (df.start < pd.Timestamp(d + b))
    df.loc[m, "label"] = nom

# Heartbleed : tous les flux vers le port 444 sont d'abord AMBIGU, sauf celui de la fenêtre officielle
hb = att & (df.dst_ip == "192.168.10.51") & (df.dst_port == 444)
df.loc[hb, "label"] = "AMBIGU"
df.loc[hb & (df.start >= pd.Timestamp(d + "18:12")) & (df.start < pd.Timestamp(d + "18:33")), "label"] = "Heartbleed"

# Tout autre flux de l'attaquant non expliqué par les règles : AMBIGU (on ne le compte pas comme normal)
reste = att & (df.label == "BENIGN")
df.loc[reste, "label"] = "AMBIGU"

print(df["label"].value_counts())
print(df[reste][["dst_ip", "dst_port"]].value_counts().head(10))

df.to_csv("/content/drive/MyDrive/nids_project/features/wednesday_labeled.csv", index=False)

In [ ]:
h = df[df.label == "DoS-Hulk"]
print(len(h), "flux Hulk")
print("SYN total vus :", h["src2dst_syn_packets"].sum())
print(h["bidirectional_packets"].describe())

In [ ]:
!pip install -q nfstream huggingface_hub
from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import hf_hub_download
local_path = hf_hub_download(
    repo_id="bencorn/CICIDS2017",
    repo_type="dataset",
    filename="pcaps/Thursday-WorkingHours.pcap",
    local_dir="/content/pcap"
)
print(local_path)

In [ ]:
from nfstream import NFStreamer
import pandas as pd

streamer = NFStreamer(
    source="/content/pcap/pcaps/Thursday-WorkingHours.pcap",
    statistical_analysis=True, n_dissections=0,
    idle_timeout=120, active_timeout=1800, n_meters=0
)
n = streamer.to_csv(path="/content/thursday_flows.csv")
print(n, "flux extraits")

df = pd.read_csv("/content/thursday_flows.csv")
df["start"] = pd.to_datetime(df["bidirectional_first_seen_ms"], unit="ms")
print(df.shape, df["start"].min(), "->", df["start"].max())
df.to_csv("/content/drive/MyDrive/nids_project/features/thursday_raw.csv", index=False)
!rm /content/pcap/pcaps/Thursday-WorkingHours.pcap

In [ ]:
# Matin : attaques web (tout part de 172.16.0.1 vers le serveur web .50)
a = df[(df.src_ip == "172.16.0.1") & (df.dst_ip == "192.168.10.50") & (df.dst_port == 80)]
pm = a.set_index("start").sort_index().resample("1min").size()
print("WEB :", " ".join(f"{t:%H:%M}={n}" for t, n in pm[pm > 0].items()))

# Après-midi : le Kali attaque Vista (.8), puis Vista scanne le réseau interne
k = df[(df.src_ip == "172.16.0.1") & (df.dst_ip == "192.168.10.8")]
pk = k.set_index("start").sort_index().resample("1min").size()
print("KALI->VISTA :", " ".join(f"{t:%H:%M}={n}" for t, n in pk[pk > 0].items()))

v = df[df.src_ip == "192.168.10.8"]
print("\nVista : destinations distinctes par tranche de 10 min")
print(v.groupby(v.start.dt.floor("10min"))["dst_ip"].nunique())

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids_project/features/thursday_raw.csv")
df["start"] = pd.to_datetime(df["start"])
d = "2017-07-06 "

fen = {"WebAttack-BruteForce": ("12:20", "13:01"),
       "WebAttack-XSS":        ("13:15", "13:36"),
       "WebAttack-SQLi":       ("13:40", "13:43")}

att = df.src_ip == "172.16.0.1"
web = att & (df.dst_ip == "192.168.10.50") & (df.dst_port == 80)

df["label"] = "BENIGN"
for nom, (a, b) in fen.items():
    m = web & (df.start >= pd.Timestamp(d + a)) & (df.start < pd.Timestamp(d + b))
    df.loc[m, "label"] = nom

# Autres flux de l'attaquant non expliqués : AMBIGU (exclus de l'entraînement et du test)
df.loc[web & (df.label == "BENIGN"), "label"] = "AMBIGU"
print(df["label"].value_counts())

In [ ]:
fin = df[(df.start >= pd.Timestamp(d + "17:00")) & (df.start < pd.Timestamp(d + "19:00"))]

# 1) Flux impliquant l'attaquant (hors attaque web du matin)
kali_ips = ["205.174.165.73", "172.16.0.1"]
k = fin[(fin.src_ip.isin(kali_ips) | fin.dst_ip.isin(kali_ips))]
print(k.groupby(["src_ip", "dst_ip", "dst_port"]).size().sort_values(ascending=False).head(15))

# 2) Quelle machine interne ouvre le plus de petits flux vers des cibles différentes ? (signature d'un scan)
petits = fin[(fin.bidirectional_packets <= 3) & fin.src_ip.str.startswith("192.168.10.")].copy()
petits["cible"] = petits.dst_ip + ":" + petits.dst_port.astype(str)
print(petits.groupby("src_ip")["cible"].nunique().sort_values(ascending=False).head(8))

In [ ]:
# 1) Petits flux de Vista, par minute, sur toute la journée
v = df[(df.src_ip == "192.168.10.8") & (df.bidirectional_packets <= 3)]
pm = v.set_index("start").sort_index().resample("1min").size()
print("VISTA petits flux :", " ".join(f"{t:%H:%M}={n}" for t, n in pm[pm > 0].items()))

# 2) Les reverse shells vers l'attaquant, port 444
r = df[(df.dst_ip == "205.174.165.73") & (df.dst_port == 444)]
print(r[["start", "src_ip", "bidirectional_duration_ms", "bidirectional_packets"]]
      .sort_values("start").to_string())

# 3) Les groupes inexpliqués (.51)
u = df[((df.src_ip == "192.168.10.51") & (df.dst_ip == "172.16.0.1") & (df.dst_port == 50122)) |
       ((df.src_ip == "172.16.0.1") & (df.dst_ip == "192.168.10.51"))]
print(len(u), "flux .51 <-> attaquant ; de", u.start.min(), "à", u.start.max())

In [ ]:
d = "2017-07-06 "
vista = "192.168.10.8"

# 1) Scan interne depuis Vista : petits flux (<= 3 paquets), minutes en rafale seulement
v = df[(df.src_ip == vista) & (df.bidirectional_packets <= 3)]
par_min = v.groupby(v.start.dt.floor("1min")).size()
minutes_scan = par_min[par_min >= 20].index
m_scan = ((df.src_ip == vista) & (df.bidirectional_packets <= 3)
          & df.start.dt.floor("1min").isin(minutes_scan))
df.loc[m_scan & (df.label == "BENIGN"), "label"] = "Scan-Vista"

# 2) Reverse shells vers l'attaquant (port 444)
c2 = (df.dst_ip == "205.174.165.73") & (df.dst_port == 444)
dans = df.start.between(pd.Timestamp(d + "17:19"), pd.Timestamp(d + "18:40"))
df.loc[c2 & dans & df.src_ip.isin([vista, "192.168.10.25"]), "label"] = "Infiltration"
df.loc[c2 & (df.label != "Infiltration"), "label"] = "AMBIGU"   # .9 à 17:13, .25 à 20:02

# 3) Tout autre flux impliquant l'attaquant et non expliqué : AMBIGU
att = (df.src_ip.isin(["172.16.0.1", "205.174.165.73"]) |
       df.dst_ip.isin(["172.16.0.1", "205.174.165.73"]))
print(((att) & (df.label == "BENIGN")).sum(), "flux de l'attaquant encore BENIGN -> AMBIGU")
df.loc[att & (df.label == "BENIGN"), "label"] = "AMBIGU"

print(df["label"].value_counts())
df.to_csv("/content/drive/MyDrive/nids_project/features/thursday_labeled.csv", index=False)

In [ ]:
!pip install -q nfstream huggingface_hub
from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import hf_hub_download
local_path = hf_hub_download(
    repo_id="bencorn/CICIDS2017",
    repo_type="dataset",
    filename="pcaps/Friday-WorkingHours.pcap",
    local_dir="/content/pcap"
)
print(local_path)

In [ ]:
from nfstream import NFStreamer
import pandas as pd

streamer = NFStreamer(
    source="/content/pcap/pcaps/Friday-WorkingHours.pcap",
    statistical_analysis=True, n_dissections=0,
    idle_timeout=120, active_timeout=1800, n_meters=0
)
n = streamer.to_csv(path="/content/friday_flows.csv")
print(n, "flux extraits")

df = pd.read_csv("/content/friday_flows.csv")
df["start"] = pd.to_datetime(df["bidirectional_first_seen_ms"], unit="ms")
print(df.shape, df["start"].min(), "->", df["start"].max())
df.to_csv("/content/drive/MyDrive/nids_project/features/friday_raw.csv", index=False)
!rm /content/pcap/pcaps/Friday-WorkingHours.pcap

In [ ]:
def par_minute(sub, titre):
    pm = sub.set_index("start").sort_index().resample("1min").size()
    print(titre, ":", " ".join(f"{t:%H:%M}={n}" for t, n in pm[pm > 0].items()))

# 1) Qui parle à l'extérieur ? (IP externes et attaquant NAT)
ext = df[~df.src_ip.str.startswith(("192.168.10.", "fe80", "ff02"))]
print(ext.groupby(["src_ip", "dst_ip"]).size().sort_values(ascending=False).head(10))

# 2) Machines internes -> attaquant (Botnet) : par machine, port et minute
bot = df[(df.dst_ip == "205.174.165.73")]
print(bot.groupby(["src_ip", "dst_port"]).size().sort_values(ascending=False).head(10))
par_minute(bot, "VERS ATTAQUANT")

# 3) Attaquant NAT -> serveur web, tous ports confondus puis port 80 seul
a = df[(df.src_ip == "172.16.0.1") & (df.dst_ip == "192.168.10.50")]
par_minute(a, "172.16.0.1->.50 (tous ports)")
par_minute(a[a.dst_port == 80], "172.16.0.1->.50:80")
print("Ports distincts visés :", a.dst_port.nunique())

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids_project/features/friday_raw.csv")
df["start"] = pd.to_datetime(df["start"])
d = "2017-07-07 "
win = lambda a, b: (df.start >= pd.Timestamp(d + a)) & (df.start < pd.Timestamp(d + b))

df["label"] = "BENIGN"
df["sous_type"] = ""

# Botnet : les 5 machines infectées -> C&C, port 8080
bots = ["192.168.10.5", "192.168.10.8", "192.168.10.9", "192.168.10.14", "192.168.10.15"]
m_bot = df.src_ip.isin(bots) & (df.dst_ip == "205.174.165.73") & (df.dst_port == 8080)
df.loc[m_bot, "label"] = "Bot"
df.loc[m_bot & win("13:02", "14:03"), "sous_type"] = "actif"
df.loc[m_bot & ~win("13:02", "14:03"), "sous_type"] = "beacon"

# PortScan et DDoS : 172.16.0.1 -> serveur web
cible = (df.src_ip == "172.16.0.1") & (df.dst_ip == "192.168.10.50")
for a, b in [("16:55", "17:01"), ("17:14", "17:20"), ("17:51", "17:57"),
             ("18:08", "18:16"), ("18:22", "18:24")]:
    df.loc[cible & win(a, b), "label"] = "PortScan"
df.loc[cible & win("18:56", "19:17"), "label"] = "DDoS"

# Tout flux de l'attaquant non expliqué : AMBIGU
att = df.src_ip.isin(["172.16.0.1", "205.174.165.73"]) | df.dst_ip.isin(["172.16.0.1", "205.174.165.73"])
print((att & (df.label == "BENIGN")).sum(), "flux AMBIGU")
df.loc[att & (df.label == "BENIGN"), "label"] = "AMBIGU"

print(df["label"].value_counts())
df.to_csv("/content/drive/MyDrive/nids_project/features/friday_labeled.csv", index=False)

In [ ]:
amb = df[df.label == "AMBIGU"]

print(amb.groupby(["src_ip", "dst_ip", "dst_port"]).size()
         .sort_values(ascending=False).head(15))

pm = amb.set_index("start").sort_index().resample("1min").size()
print(" ".join(f"{t:%H:%M}={n}" for t, n in pm[pm > 0].items()))

print(amb["bidirectional_packets"].describe())

In [ ]:
# Reprendre après l'étiquetage du vendredi (df en mémoire)
inv = (df.label == "AMBIGU") & (df.src_ip == "192.168.10.50") & (df.dst_ip == "172.16.0.1")

for a, b in [("16:55", "17:01"), ("17:14", "17:20"), ("17:51", "17:57"),
             ("18:08", "18:16"), ("18:22", "18:24")]:
    m = inv & win(a, b)
    df.loc[m, "label"] = "PortScan"; df.loc[m, "sous_type"] = "reverse"

m = inv & win("18:56", "19:17")
df.loc[m, "label"] = "DDoS"; df.loc[m, "sous_type"] = "reverse"

print(df["label"].value_counts())
print(df[df.sous_type == "reverse"]["label"].value_counts())
df.to_csv("/content/drive/MyDrive/nids_project/features/friday_labeled.csv", index=False)


In [ ]:
import json, numpy as np, pandas as pd
base = "/content/drive/MyDrive/nids_project"
feature_cols = json.load(open(f"{base}/models/feature_cols.json"))
keep = set(feature_cols) | {"src_ip", "dst_ip", "dst_port", "start", "label", "sous_type"}

parts = []
for jour in ["monday", "tuesday", "wednesday", "thursday", "friday"]:
    p = pd.read_csv(f"{base}/features/{jour}_labeled.csv", usecols=lambda c: c in keep)
    p["jour"] = jour
    parts.append(p)
df = pd.concat(parts, ignore_index=True)
df["start"] = pd.to_datetime(df["start"])

print(df.groupby(["jour", "label"]).size().to_string())
print("\nTOTAL :", len(df))

# Scan-Vista vs PortScan : mêmes features ?
cols = ["bidirectional_packets", "bidirectional_bytes", "bidirectional_duration_ms",
        "bidirectional_syn_packets", "bidirectional_rst_packets", "bidirectional_mean_ps"]
print(df[df.label.isin(["Scan-Vista", "PortScan"])].groupby("label")[cols].median().T)

In [ ]:
import json, numpy as np, pandas as pd

# 1) Classes finales
df["label"] = df["label"].replace({"WebAttack-BruteForce": "WebAttack",
                                   "WebAttack-XSS": "WebAttack",
                                   "WebAttack-SQLi": "WebAttack"})
df = df[~df["label"].isin(["AMBIGU", "Heartbleed", "Infiltration"])].reset_index(drop=True)

classes = ["BENIGN"] + sorted(c for c in df["label"].unique() if c != "BENIGN")
json.dump(classes, open(f"{base}/models/classes.json", "w"))
print(classes)

# 2) Split chronologique par (classe, jour)
df["minute"] = df["start"].dt.floor("1min").astype("int64")
grp = df.groupby(["label", "jour"])["minute"]
r = grp.rank(method="dense")            # rang de la minute dans son groupe
n = grp.transform("nunique")
a, b = np.floor(0.60 * n), np.floor(0.80 * n)

df["split"] = np.select([r <= a, (r > a + 1) & (r <= b), r > b + 1],
                        ["train", "val", "test"], default="gap")
df = df[df["split"] != "gap"]
print(pd.crosstab(df["label"], df["split"]))

# 3) Plafonner les grosses classes DANS LE TRAIN uniquement
CAP = 40_000
tr = df[df.split == "train"].groupby("label", group_keys=False).apply(
        lambda g: g.sample(min(len(g), CAP), random_state=42))
print(tr["label"].value_counts())

# 4) Sauvegarde (jeu complet avec la colonne split)
df.to_parquet(f"{base}/features/dataset_5jours.parquet", index=False)
